In [69]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [70]:
# Material Parameters
mu = 0.8
jm = 40.0
Identity = torch.eye(3)

In [71]:
# Gent Stress Function
def gent_stress_function(lamda, mu, jm):
    I1 = lamda**2 + 2/lamda
    stress = mu * (lamda - 1/lamda**2) / (1 - (I1 - 3)/jm)
    return stress

In [72]:
def deformation_gradient(lamda):
    diagonal_values = torch.stack([
        lamda,
        lamda**(-0.5),
        lamda**(-0.5)
    ], dim=-1)

    return torch.diag_embed(diagonal_values)

In [73]:
def frobenius_norm(F):
    return torch.sqrt(
        torch.sum((F - Identity)**2, dim=(-2, -1))
    )

In [74]:
def add_noise(P_true_train,eta_min,eta_max,lamda):
    P_char = torch.max(torch.abs(P_true_train))
    noise_sd_min = eta_min * P_char
    noise_sd_max = eta_max * P_char
    F = deformation_gradient(lamda)
    F_lambdamax = deformation_gradient(torch.max(lamda))
    q = 2.0
    t = frobenius_norm(F)
    ksy = torch.randn_like(P_true_train)
    conditional_noise_variance = torch.square(noise_sd_min) + (torch.square(noise_sd_max) - torch.square(noise_sd_min)) * ((t / frobenius_norm(F_lambdamax))**q)
    return torch.sqrt(conditional_noise_variance) * ksy

In [75]:
# Synthetic Data Generation
def generate_synthetic_data(num_samples, lamda_min, lamda_max,eta_min,eta_max):
    lamda_train = torch.linspace(lamda_min, lamda_max, num_samples)
    I1_train = lamda_train**2 + 2/lamda_train
    P_true_train = gent_stress_function(lamda_train, mu, jm)   
    noise = add_noise(P_true_train,eta_min,eta_max,lamda_train)
    Y_train = P_true_train + noise
    Y_train = Y_train.reshape(-1, 1)
    lamda_train =lamda_train.reshape(-1, 1)
    I1_train = I1_train.reshape(-1, 1)
    x = torch.cat((lamda_train, I1_train,Y_train), dim=1)
    return x


In [76]:
x = generate_synthetic_data(num_samples=50000, lamda_min=1.0, lamda_max=4.0, eta_min=0.005,eta_max=0.03)

In [77]:
def noise(u):
    return torch.log(u) - torch.log(1-u)

In [78]:
class NeuralNetwork(nn.Module):
    def __init__(self,m = 10):
        super().__init__()
        self.a = nn.Parameter(torch.randn(1))
        self.c = nn.Parameter(torch.randn(m))
        self.w = nn.Parameter(torch.randn(m))
        self.b = nn.Parameter(torch.randn(m))

        # Extreme Sparsification:
        self.beta = 2/3

    

        self.alpha_a = nn.Parameter(torch.randn(1))
        self.alpha_c = nn.Parameter(torch.randn(m))
        self.alpha_w = nn.Parameter(torch.randn(m))
        self.alpha_b = nn.Parameter(torch.randn(m))   

        

        self.gamma = -0.1
        self.zeta = 1.1  

    
    def forward(self,I1):


        u_a = torch.rand_like(self.a)
        u_c = torch.rand_like(self.c)
        u_w = torch.rand_like(self.w)
        u_b = torch.rand_like(self.b)

        eps = 1e-6

        u_a = torch.clamp(u_a, eps, 1-eps)
        u_c = torch.clamp(u_c, eps, 1-eps)
        u_w = torch.clamp(u_w, eps, 1-eps)
        u_b = torch.clamp(u_b, eps, 1-eps)

        s_a = torch.sigmoid((noise(u_a) + (self.alpha_a) )/self.beta)
        s_c = torch.sigmoid((noise(u_c) + (self.alpha_c) )/self.beta)
        s_w = torch.sigmoid((noise(u_w) + (self.alpha_w) )/self.beta)
        s_b = torch.sigmoid((noise(u_b) + (self.alpha_b) )/self.beta)

        s_bar_a = s_a*(self.zeta-self.gamma) +  self.gamma
        s_bar_c = s_c*(self.zeta-self.gamma) +  self.gamma
        s_bar_w = s_w*(self.zeta-self.gamma) +  self.gamma
        s_bar_b = s_b*(self.zeta-self.gamma) +  self.gamma
        
                
        a_mask = torch.clamp(s_bar_a,0,1)
        c_mask = torch.clamp(s_bar_c,0,1)
        w_mask = torch.clamp(s_bar_w,0,1)
        b_mask = torch.clamp(s_bar_b,0,1)




        a = F.softplus(self.a )* a_mask
        c = F.softplus(self.c )* c_mask
        w = F.softplus(self.w )* w_mask
        b = self.b * b_mask
        x = I1 - 3

        z = x * w + b

        # Broadcasting (element wise multiplication)

        hidden = (F.softplus(z) - F.softplus(b))
        psi = (a * x + torch.sum(c * hidden, dim = -1, keepdim = True))
        # sum across the last dimension to get the final output (sum across the columns)
        return psi

In [79]:
def weighted_loss(model,data,noise_variance,gamma,zeta,beta, alpha_a, alpha_c, alpha_w, alpha_b):
    lamda = data[:,0].reshape(-1,1)
    I1 = data[:,1].reshape(-1,1)
    P_true = data[:,2].reshape(-1,1)
    I1.requires_grad_(True)

    rho = 0.1
    prob_a = torch.sigmoid(alpha_a - beta * torch.log(torch.tensor(-gamma/zeta)))  
    prob_c = torch.sigmoid(alpha_c - beta * torch.log(torch.tensor(-gamma/zeta)))  
    prob_w = torch.sigmoid(alpha_w - beta * torch.log(torch.tensor(-gamma/zeta)))  
    prob_b = torch.sigmoid(alpha_b - beta * torch.log(torch.tensor(-gamma/zeta)))

    sparsity_penalty = rho * (torch.sum(prob_a) + torch.sum(prob_c) + torch.sum(prob_w) + torch.sum(prob_b))

    P_pred = model(I1)
    # Compute the gradient of the predicted energy potential with respect to I1
    dW_dI1 = torch.autograd.grad(P_pred, I1, grad_outputs=torch.ones_like(P_pred), create_graph=True)[0]
    P_pred = 2 * dW_dI1 * (lamda - 1/lamda**2)

    squared_error= (P_pred - P_true)**2
    if noise_variance is None:
        loss = torch.mean(squared_error)
    else:
        loss = torch.mean(squared_error / noise_variance)

    loss += sparsity_penalty
    return loss,P_pred

In [80]:
model = NeuralNetwork(m=32)

In [81]:
def Backpropagation(model, data, noise_variance,gamma,zeta,beta, learning_rate=0.01,num_epochs=1):
    model = NeuralNetwork(m=32)
    

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(num_epochs):
      optimizer.zero_grad()
      loss,P_pred = weighted_loss(model, data, noise_variance,gamma=gamma, zeta=zeta, beta=beta, alpha_a=model.alpha_a, alpha_c=model.alpha_c, alpha_w=model.alpha_w, alpha_b=model.alpha_b)
    
      loss.backward()
      optimizer.step()
    
      if epoch % 100 == 0:
          print(f'Epoch [{epoch}/{num_epochs}], Loss: {loss.item():.10f}')  

In [82]:
Backpropagation(model, x, noise_variance=100,gamma=-0.1, zeta=1.1, beta=2/3, learning_rate=0.01,num_epochs=1000)

Epoch [0/1000], Loss: 10.1199436188
Epoch [100/1000], Loss: 6.3698081970
Epoch [200/1000], Loss: 4.8815274239
Epoch [300/1000], Loss: 3.5539600849
Epoch [400/1000], Loss: 2.5830168724
Epoch [500/1000], Loss: 2.0060033798
Epoch [600/1000], Loss: 1.5621687174
Epoch [700/1000], Loss: 1.2509151697
Epoch [800/1000], Loss: 1.1033577919
Epoch [900/1000], Loss: 0.8901333213
